# 2 · DPO (Direct Preference Optimization)
**Idea:** for each request, show a good answer and a bad answer. The model learns to prefer the good one.

`black sneakers under 50 dollars` → `FIND shoes color=BK max=50`

Here the prompt explains the rules, so the model is already close. But it still makes typical mistakes,
such as writing the color name instead of the code. DPO trains those mistakes away.

## Step 1 · Load a small model

In [10]:
import warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings('ignore')            # hide technical warnings

MODEL = 'Qwen/Qwen3.5-0.8B'                  # small open model (0.8 billion parameters)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 324.79it/s]


## Step 2 · Write the examples
Each example has a request, a good answer and a bad answer.
The bad answers show typical mistakes: the color name instead of the code, the wrong item name, a missing price, or extra words.

In [12]:
TRAIN = [  # (request, good answer, bad answer)
    ('red tee',                           'FIND shirt color=RD',         'FIND tee color=RD'),
    ('blue shirt',                        'FIND shirt color=BL',         'FIND shirt color=blue'),
    ('black t-shirt under 20 dollars',    'FIND shirt color=BK max=20',  'FIND shirt color=BK'),
    ('white tee below 30',                'FIND shirt color=WH max=30',  'Here is the command: FIND shirt color=WH max=30'),
    ('I want a red t-shirt',              'FIND shirt color=RD',         'FIND shirt color=red'),
    ('white shirt',                       'FIND shirt color=WH',         'Here is the command: FIND shirt color=WH'),
    ('black sneakers',                    'FIND shoes color=BK',         'FIND sneakers color=BK'),
    ('white trainers under 80 dollars',   'FIND shoes color=WH max=80',  'FIND shoes color=white max=80'),
    ('blue shoes',                        'FIND shoes color=BL',         'Here is the command: FIND shoes color=BL'),
    ('red sneakers below 50',             'FIND shoes color=RD max=50',  'FIND shoes color=RD'),
    ('show me black trainers',            'FIND shoes color=BK',         'FIND trainers color=BK'),
    ('red shoes under 30 dollars',        'FIND shoes color=RD max=30',  'FIND shoes color=red max=30'),
    ('blue trainers',                     'FIND shoes color=BL',         'FIND trainers color=BL'),
    ('black coat',                        'FIND jacket color=BK',        'FIND coat color=BK'),
    ('blue jacket under 80 dollars',      'FIND jacket color=BL max=80', 'FIND jacket color=blue max=80'),
    ('white coat below 50',               'FIND jacket color=WH max=50', 'FIND jacket color=WH'),
    ('I want a red jacket',               'FIND jacket color=RD',        'Here is the command: FIND jacket color=RD'),
    ('blue coat',                         'FIND jacket color=BL',        'FIND coat color=blue'),
    ('black jacket below 80',             'FIND jacket color=BK max=80', 'FIND jacket color=BK'),
    ('red hoodie',                        'FIND hoodie color=RD',        'FIND hoodie color=red'),
    ('white sweatshirt under 30 dollars', 'FIND hoodie color=WH max=30', 'FIND sweatshirt color=WH max=30'),
    ('black hoodie below 20',             'FIND hoodie color=BK max=20', 'FIND hoodie color=BK'),
    ('show me a blue sweatshirt',         'FIND hoodie color=BL',        'Here is the command: FIND hoodie color=BL'),
    ('white hoodie',                      'FIND hoodie color=WH',        'FIND hoodie color=white'),
    ('red sweatshirt below 50',           'FIND hoodie color=RD max=50', 'FIND sweatshirt color=RD max=50'),
]

TEST = [
    ('blue tee under 50 dollars',         'FIND shirt color=BL max=50'),
    ('white sneakers',                    'FIND shoes color=WH'),
    ('red coat below 30',                 'FIND jacket color=RD max=30'),
    ('black sweatshirt',                  'FIND hoodie color=BK'),
    ('show me a white t-shirt',           'FIND shirt color=WH'),
]

RULES = '''Write the search command for the shop request.
Format: FIND <item> color=<code> max=<price>   (max only if a price is given)
Items: shirt (shirt, tee, t-shirt), shoes (shoes, sneakers, trainers), jacket (jacket, coat), hoodie (hoodie, sweatshirt)
Color codes: red=RD, blue=BL, black=BK, white=WH
Reply with only the command.'''

def question(request):
    return [{'role': 'user', 'content': f'{RULES}\n\nRequest: {request}'}]

## Step 3 · Ask the model before training
The prompt explains every rule. **Look for** the kinds of mistakes it still makes.

In [13]:
def ask(model, request):
    inputs = tokenizer.apply_chat_template(question(request), add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
    output = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    answer = output[0][inputs['input_ids'].shape[1]:]   # keep only the new words
    return tokenizer.decode(answer, skip_special_tokens=True).strip()

def test(model):
    model.eval() # answering mode
    right = 0
    for request, command in TEST:
        answer = ask(model, request)
        if answer == command:
            right += 1
            print('right:', request, '->', answer)
        else:
            print('wrong:', request, '->', answer, '   expected:', command)
    print(right, 'of', len(TEST), 'right')

test(model)

right: blue tee under 50 dollars -> FIND shirt color=BL max=50
wrong: white sneakers -> FIND shoes color=WH max=100    expected: FIND shoes color=WH
right: red coat below 30 -> FIND jacket color=RD max=30
wrong: black sweatshirt -> FIND black sweatshirt color=BL max=100    expected: FIND hoodie color=BK
wrong: show me a white t-shirt -> FIND shirt color=WH max=10    expected: FIND shirt color=WH
2 of 5 right


## Step 4 · Train
Each example: the prompt, the good answer (`chosen`) and the bad answer (`rejected`).
The model makes good answers more likely and bad answers less likely, compared with how it started.

- **beta:** how far the model may move away from where it started. Small = careful.
- **Watch:** `rewards/accuracies` going to 1.0 means the good answer wins in every pair.

In [14]:
from datasets import Dataset
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

data = Dataset.from_list([
    {'prompt': question(request),
     'chosen': [{'role': 'assistant', 'content': good}],
     'rejected': [{'role': 'assistant', 'content': bad}]}
    for request, good, bad in TRAIN
])

trainer = DPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=data,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=DPOConfig(
        beta=0.1,                # stay close to the starting model
        num_train_epochs=8,     # each example is seen 10 times
        learning_rate=1e-5,      # size of each small adjustment
        logging_steps=1,         # print progress every 4 steps
        save_strategy='no',      # do not save copies to disk
    ),
)
trainer.train()

Tokenizing train dataset: 100%|██████████| 25/25 [00:00<00:00, 1080.24 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 25/25 [00:00<00:00, 10312.51 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
1,0.693147
2,0.687848
3,0.677418
4,0.659934
5,0.619076
6,0.581171
7,0.582118
8,0.595569
9,0.495037
10,0.494886


TrainOutput(global_step=32, training_loss=0.36384187219664454, metrics={'train_runtime': 34.0457, 'train_samples_per_second': 5.874, 'train_steps_per_second': 0.94, 'total_flos': 160626308832000.0, 'train_loss': 0.36384187219664454, 'epoch': 8.0})

## Step 5 · Ask the model after training
The same 5 test requests. **Look for** fewer of the typical mistakes.

In [15]:
test(trainer.model)

right: blue tee under 50 dollars -> FIND shirt color=BL max=50
wrong: white sneakers -> FIND shoes color=WH max=0    expected: FIND shoes color=WH
right: red coat below 30 -> FIND jacket color=RD max=30
wrong: black sweatshirt -> FIND shirt color=BK max=100    expected: FIND hoodie color=BK
wrong: show me a white t-shirt -> FIND shirt color=WH max=0    expected: FIND shirt color=WH
2 of 5 right
